In [1]:
# ==========================================================
# STUDENT 3
# NOTEBOOK 01
# BERT EMBEDDING EXTRACTION
# ==========================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    BertModel,
    DataCollatorWithPadding
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

In [2]:
# ==========================================================
# CONFIGURATION
# ==========================================================

MODEL_NAME = "bert-base-uncased"

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

NOTEBOOK_DIR = Path.cwd()

PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

PROCESSED_DATA = PROJECT_ROOT / "datasets" / "processed"

MODELS = PROJECT_ROOT / "models"

EMBEDDINGS = PROJECT_ROOT / "embeddings"

EMBEDDINGS.mkdir(exist_ok=True)

MAX_LENGTH = 512

BATCH_SIZE = 8

print("="*70)
print("BERT EMBEDDING EXTRACTION")
print("="*70)

print(device)

BERT EMBEDDING EXTRACTION
cpu


In [3]:
train_df = pd.read_csv(
    PROCESSED_DATA /
    "bert_train_subset.csv"
)

validation_df = pd.read_csv(
    PROCESSED_DATA /
    "bert_validation_subset.csv"
)

test_df = pd.read_csv(
    PROCESSED_DATA /
    "bert_test_subset.csv"
)

print(train_df.shape)
print(validation_df.shape)
print(test_df.shape)

(5000, 7)
(1000, 7)
(1000, 7)


In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [5]:
# ==========================================================
# EMAIL DATASET
# ==========================================================

class EmailDataset(Dataset):

    def __init__(self, dataframe, tokenizer):

        self.data = dataframe.reset_index(drop=True)

        self.tokenizer = tokenizer

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        text = str(
            self.data.loc[idx, "cleaned_email_text"]
        )

        label = int(
            self.data.loc[idx, "label"]
        )

        encoding = self.tokenizer(

            text,

            truncation=True,

            max_length=MAX_LENGTH,

            padding=False,

            return_tensors="pt"

        )

        item = {

            "input_ids":
                encoding["input_ids"].squeeze(0),

            "attention_mask":
                encoding["attention_mask"].squeeze(0),

            "labels":
                torch.tensor(
                    label,
                    dtype=torch.long
                )

        }

        return item

print("="*70)
print("EmailDataset Ready")
print("="*70)

EmailDataset Ready


In [6]:
# ==========================================================
# CREATE DATALOADERS
# ==========================================================

collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

train_dataset = EmailDataset(
    train_df,
    tokenizer
)

validation_dataset = EmailDataset(
    validation_df,
    tokenizer
)

test_dataset = EmailDataset(
    test_df,
    tokenizer
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator
)

print("=" * 70)
print("DATALOADERS CREATED")
print("=" * 70)

print("Training batches   :", len(train_loader))
print("Validation batches :", len(validation_loader))
print("Test batches       :", len(test_loader))


DATALOADERS CREATED
Training batches   : 625
Validation batches : 125
Test batches       : 125


In [7]:
# ==========================================================
# LOAD FINE-TUNED BERT ENCODER
# ==========================================================

from transformers import BertForSequenceClassification

classifier = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

classifier.load_state_dict(
    torch.load(
        MODELS / "best_bert_model.pt",
        map_location=device
    )
)

classifier.to(device)

classifier.eval()

# ----------------------------------------------------------
# Extract only the encoder
# ----------------------------------------------------------

bert_encoder = classifier.bert

bert_encoder.eval()

print("=" * 70)
print("FINE-TUNED BERT ENCODER LOADED")
print("=" * 70)

print(type(bert_encoder).__name__)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FINE-TUNED BERT ENCODER LOADED
BertModel


In [9]:
# ==========================================================
# GENERIC EMBEDDING EXTRACTION FUNCTION
# ==========================================================

from tqdm.auto import tqdm

def extract_embeddings(model, dataloader, dataset_name):

    embeddings = []
    labels = []

    model.eval()

    with torch.no_grad():

        for batch in tqdm(
            dataloader,
            desc=f"Extracting {dataset_name}"
        ):

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask

            )

            # CLS Token Representation
            cls_embeddings = outputs.last_hidden_state[:, 0, :]

            embeddings.append(
                cls_embeddings.cpu().numpy()
            )

            labels.append(
                batch["labels"].numpy()
            )

    embeddings = np.vstack(embeddings)

    labels = np.concatenate(labels)

    print("=" * 70)
    print(f"{dataset_name.upper()} COMPLETED")
    print("=" * 70)
    print("Embeddings :", embeddings.shape)
    print("Labels     :", labels.shape)

    return embeddings, labels

In [10]:
train_embeddings, train_labels = extract_embeddings(

    bert_encoder,

    train_loader,

    "Training"

)

Extracting Training:   0%|          | 0/625 [00:00<?, ?it/s]

TRAINING COMPLETED
Embeddings : (5000, 768)
Labels     : (5000,)


In [11]:
train_embeddings, train_labels = extract_embeddings(

    bert_encoder,

    train_loader,

    "Training"

)

Extracting Training:   0%|          | 0/625 [00:01<?, ?it/s]

TRAINING COMPLETED
Embeddings : (5000, 768)
Labels     : (5000,)


In [12]:
validation_embeddings, validation_labels = extract_embeddings(

    bert_encoder,

    validation_loader,

    "Validation"

)

Extracting Validation:   0%|          | 0/125 [00:00<?, ?it/s]

VALIDATION COMPLETED
Embeddings : (1000, 768)
Labels     : (1000,)


In [13]:
test_embeddings, test_labels = extract_embeddings(

    bert_encoder,

    test_loader,

    "Test"

)

Extracting Test:   0%|          | 0/125 [00:00<?, ?it/s]

TEST COMPLETED
Embeddings : (1000, 768)
Labels     : (1000,)


In [14]:
# ==========================================================
# SAVE BERT EMBEDDINGS
# ==========================================================

BERT_EMBEDDINGS = EMBEDDINGS / "bert"

BERT_EMBEDDINGS.mkdir(
    parents=True,
    exist_ok=True
)

np.save(
    BERT_EMBEDDINGS / "train_embeddings.npy",
    train_embeddings
)

np.save(
    BERT_EMBEDDINGS / "validation_embeddings.npy",
    validation_embeddings
)

np.save(
    BERT_EMBEDDINGS / "test_embeddings.npy",
    test_embeddings
)

np.save(
    BERT_EMBEDDINGS / "train_labels.npy",
    train_labels
)

np.save(
    BERT_EMBEDDINGS / "validation_labels.npy",
    validation_labels
)

np.save(
    BERT_EMBEDDINGS / "test_labels.npy",
    test_labels
)

print("=" * 70)
print("BERT EMBEDDINGS SAVED SUCCESSFULLY")
print("=" * 70)

BERT EMBEDDINGS SAVED SUCCESSFULLY
